# 9. Risk Management: Value at Risk (VaR) & Expected Shortfall (CVaR) - Interactive

**Objective**: Understand how to quantify and manage portfolio risk using Value at Risk and Expected Shortfall.

## What You'll Learn
- **Value at Risk (VaR)**: Maximum expected loss at a given confidence level
- **Expected Shortfall (CVaR)**: Average loss when VaR is breached (tail risk)
- **Risk Sensitivities**: How volatility, time horizon, and portfolio size affect risk
- **Why 2008 Happened**: Limitations of VaR and importance of tail risk
- **Real-world Risk Management**: How banks and funds use these metrics

## Key Formula
$$\text{VaR}_{95\%} = S_0 - \text{Percentile}(S_T, 5\%)$$

$$\text{CVaR}_{95\%} = S_0 - E[S_T | S_T < \text{VaR}_{95\%}]$$

In [1]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Output
    widgets_available = True
except ImportError:
    widgets_available = False
    print("⚠ ipywidgets not available. Please install: pip install ipywidgets")

%matplotlib inline

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [2]:
# Define VaR Calculation Functions

def calculate_var_cvar(S0, mu, sigma, T, N_sim, confidence_level=95):
    """
    Calculate Value at Risk and Conditional Value at Risk.
    
    Parameters:
    - S0: Initial portfolio value
    - mu: Expected annual return
    - sigma: Annual volatility
    - T: Time horizon (years)
    - N_sim: Number of simulations
    - confidence_level: Confidence level (90, 95, 99)
    
    Returns:
    - ST: Array of final portfolio values
    - VaR: Value at Risk (dollar loss)
    - CVaR: Expected Shortfall (average loss in tail)
    - var_pct: VaR as percentage
    - cvar_pct: CVaR as percentage
    """
    np.random.seed(42)
    
    # Generate portfolio values using GBM
    Z = np.random.standard_normal(N_sim)
    ST = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Calculate VaR at confidence level
    percentile = 100 - confidence_level
    var_threshold = np.percentile(ST, percentile)
    VaR = S0 - var_threshold
    var_pct = (VaR / S0) * 100
    
    # Calculate CVaR (average of tail events)
    tail_events = ST[ST < var_threshold]
    CVaR = S0 - np.mean(tail_events)
    cvar_pct = (CVaR / S0) * 100
    
    return ST, VaR, CVaR, var_pct, cvar_pct


def display_risk_analysis(S0=100000, mu=0.08, sigma=0.2, T=1, N_sim=50000, confidence=95):
    """
    Display VaR and CVaR analysis with visualizations.
    """
    
    ST, VaR, CVaR, var_pct, cvar_pct = calculate_var_cvar(S0, mu, sigma, T, N_sim, confidence)
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left plot: VaR Distribution
    counts, bins, patches = ax1.hist(ST, bins=100, color='skyblue', edgecolor='black', alpha=0.7)
    
    var_threshold = np.percentile(ST, 100 - confidence)
    for i, patch in enumerate(patches):
        if bins[i] < var_threshold:
            patch.set_facecolor('salmon')
    
    ax1.axvline(var_threshold, color='red', linestyle='--', linewidth=2.5, 
                label=f'VaR ({confidence}%): ${VaR:,.0f}\n({var_pct:.2f}% loss)')
    ax1.axvline(S0, color='green', linestyle='-', linewidth=2, alpha=0.5, 
               label=f'Initial: ${S0:,.0f}')
    ax1.set_xlabel('Portfolio Value ($)', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title(f'Portfolio Distribution ({T:.1f}y, σ={sigma:.1%})', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Right plot: VaR vs CVaR Comparison
    tail_events = ST[ST < var_threshold]
    ax2.hist(ST, bins=100, color='lightblue', edgecolor='black', alpha=0.5)
    ax2.hist(tail_events, bins=50, color='crimson', edgecolor='darkred', alpha=0.8, label='Tail Events')
    ax2.axvline(var_threshold, color='red', linestyle='--', linewidth=2.5, 
                label=f'VaR: ${VaR:,.0f}')
    ax2.axvline(np.mean(tail_events), color='darkred', linestyle='-', linewidth=2.5, 
                label=f'CVaR: ${CVaR:,.0f}\n(Avg Tail Loss)')
    ax2.set_xlabel('Portfolio Value ($)', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title(f'VaR vs CVaR: Gap = ${CVaR - VaR:,.0f}\n(Underestimation Risk)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n" + "="*70)
    print(f"PORTFOLIO RISK ANALYSIS")
    print("="*70)
    print(f"Initial Portfolio Value:      ${S0:,.0f}")
    print(f"Time Horizon:                 {T:.3f} years ({T*365:.0f} days)")
    print(f"Expected Annual Return:       {mu:.2%}")
    print(f"Annual Volatility:            {sigma:.2%}")
    print(f"Simulations Run:              {N_sim:,}")
    print("-"*70)
    print(f"\nValue at Risk ({confidence}% Confidence):")
    print(f"  Dollar Loss:                ${VaR:,.2f}")
    print(f"  Percentage Loss:            {var_pct:.2f}%")
    print(f"  Expected Portfolio Value:   ${var_threshold:,.0f}")
    print(f"  Interpretation:             {confidence}% chance loss is LESS than this")
    print(f"                              {100-confidence}% chance loss is MORE than this")
    print("-"*70)
    print(f"\nConditional VaR (Expected Shortfall):")
    print(f"  Dollar Loss:                ${CVaR:,.2f}")
    print(f"  Percentage Loss:            {cvar_pct:.2f}%")
    print(f"  Expected Value If Breached: ${np.mean(tail_events):,.0f}")
    print(f"  Interpretation:             AVERAGE loss when worst {100-confidence}% happens")
    print("-"*70)
    print(f"\nRisk Gap Analysis:")
    print(f"  CVaR - VaR:                 ${CVaR - VaR:,.2f}")
    print(f"  Gap Percentage:             {((CVaR - VaR)/VaR)*100:.1f}% worse than VaR")
    print(f"  Why It Matters:             VaR alone UNDERESTIMATES true risk")
    print(f"  Real-world Implication:     When crisis hits, losses average ${CVaR:,.0f}")
    print("="*70 + "\n")

print("✓ Risk analysis functions defined")

✓ Risk analysis functions defined


## Interactive Risk Analysis Explorer

**Use the sliders below to explore how portfolio risk changes with different market conditions:**

- **Portfolio Value (S₀)**: Amount of money you're managing [$50k-$500k]
- **Expected Return (μ)**: What you expect the portfolio to earn annually [0%-20%]
- **Volatility (σ)**: Market uncertainty/fluctuations [5%-50%]
- **Time Horizon (T)**: How long until you need the money [1-5 years]
- **Confidence Level**: What "worst case" means to you [90%-99%]
- **Simulations**: How many scenarios to run [1k-100k]

**Watch how:**
- ↑ Volatility → ↑ Risk (VaR/CVaR increase)
- ↑ Longer time horizon → ↑ Risk (more time for big moves)
- Higher confidence (99% vs 90%) → Higher risk estimate
- CVaR always ≥ VaR (tail risk is always scary)

In [3]:
# Interactive Parameter Exploration
if widgets_available:
    interact(
        display_risk_analysis,
        S0=FloatSlider(value=100000, min=50000, max=500000, step=10000, description='Portfolio (S₀)', style={'description_width': '130px'}),
        mu=FloatSlider(value=0.08, min=0.0, max=0.2, step=0.01, description='Return (μ)', style={'description_width': '130px'}),
        sigma=FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description='Volatility (σ)', style={'description_width': '130px'}),
        T=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Time Horizon (T)', style={'description_width': '130px'}),
        N_sim=IntSlider(value=50000, min=1000, max=100000, step=5000, description='Simulations', style={'description_width': '130px'}),
        confidence=FloatSlider(value=95, min=90, max=99, step=1, description='Confidence %', style={'description_width': '130px'})
    )
else:
    print("⚠ Interactive sliders not available. Using static visualization instead.")
    display_risk_analysis(100000, 0.08, 0.2, 1, 50000, 95)

interactive(children=(FloatSlider(value=100000.0, description='Portfolio (S₀)', max=500000.0, min=50000.0, ste…

## Guided Learning Experiments

### Experiment 1: Volatility Impact (Most Important!)
- Keep everything at default except σ
- Increase σ from 5% to 50%
- **Observation:** VaR grows MUCH faster than you'd expect
- **Real-world:** This is why in 2008, banks didn't realize how much risk they had—volatility exploded

### Experiment 2: Portfolio Size Scales Risk
- Keep σ, T, μ constant
- Try S₀ = $50k, $100k, $500k
- **Observation:** Larger portfolio → larger absolute dollar loss (but same % loss)
- **Real-world:** Absolute losses matter when you're the bank and need cash

### Experiment 3: Time Horizon Extension
- Keep S₀=$100k, σ=20%, μ=8%
- Increase T from 1 year to 5 years
- **Observation:** Longer time = higher potential moves = higher VaR
- **Real-world:** 401k investors have 30 years to recover; day traders have hours

### Experiment 4: Confidence Level Trade-Off
- Use default parameters
- Compare 90%, 95%, 99% confidence
- **Observation:** Going from 95% to 99% is not just 4% more risk—it's often 50%+ more
- **Real-world:** Regulators require 99% VaR. This costs banks money in reserves.

### Experiment 5: When CVaR Matters Most
- Set σ = 50% (high volatility market)
- Look at the gap between VaR and CVaR
- Compare to σ = 10% (calm market)
- **Observation:** In turbulent markets, CVaR >> VaR (tail events are SEVERE)
- **Real-world:** COVID crash, 2008 crisis, 1987 Black Monday—all had huge CVaR gaps

### Experiment 6: 2008 Crisis Simulation
- Set σ = 40% (market panic), T = 0.5 years (short term pain)
- Set confidence = 99% (worst case)
- **Observation:** Massive losses on 1% tail events
- **Real-world:** Banks thought 99% VaR was safe. It wasn't—2008 proved it.

## Key Insights: When & Why Risk Explodes

### Understanding VaR vs CVaR

| Metric | What It Is | When to Use | Limitation |
|--------|-----------|-------------|-------------|
| **VaR** | Worst loss at X% confidence | Daily risk monitoring | Ignores tail severity |
| **CVaR** | Average loss WHEN worst happens | Stress testing, regulation | More complex to calculate |

### Real-World Example: $100k Portfolio
- **Normal market (σ=15%):**
  - 95% VaR = $8k loss
  - 95% CVaR = $9k loss
  - Gap = Only $1k (tail events aren't too severe)

- **Crisis market (σ=40%):**
  - 95% VaR = $22k loss
  - 95% CVaR = $27k loss
  - Gap = $5k (tail events ARE severe!)

### Why This Matters

1. **VaR is Deceptive:**
   - "95% VaR = $8k loss" sounds manageable
   - But when 5% probability hits, average loss = $27k
   - YOU WERE UNPREPARED

2. **The 2008 Financial Crisis:**
   - Banks calculated 99% VaR
   - Thought: "Only 1% chance of crisis"
   - Forgot: When that 1% happens, losses are EXTREME
   - CVaR was 3-5x higher than VaR
   - Banks failed because they used VaR alone

3. **Modern Risk Management:**
   - Regulators now require BOTH VaR and CVaR
   - Stress tests with extreme scenarios
   - FAT TAIL risk estimation (extreme events matter)

### The Bottom Line

| Question | Answer | Why |
|----------|--------|-----|
| What's the most likely loss? | VaR percentile tells you | It's the threshold, not average |
| What if the worst 1% happens? | CVaR tells you the true cost | Average loss in tail events |
| How do I stay safe? | Use CVaR + scenario analysis | Don't trust VaR alone |
| Why do banks hold capital? | To cover CVaR losses | VaR gaps exist, they're expensive |

### Historical Volatility Reference
- Normal stock market: σ = 15-20%
- Pre-crisis: σ = 12-15% ("goldilocks zone")
- 2008 Crisis peak: σ = 40-50%
- COVID crash (March 2020): σ = 35-45%
- During crisis, CVaR can be 50-200% higher than VaR